# Explain Step Tutorial


## What you'll learn
1. Theory Primer <br>
2. How to run explain step <br>
3. How to customize the explain step <br>
4. Using LLMs to get biological insights of the latent dimensions <br>

### 1) Theory Primer
TODO

### 2) How To Perform the Explain Step
First, we need to run an AUTOENCODIX pipeline, `Ontix` is a very good choice here.
#### ❗❗ Requirements: Getting Tutorial Data ❗❗
To follow along, please download the date from the link below (1GB):

https://cloud.scadsai.uni-leipzig.de/index.php/s/QXYnieKY8AA3Zta/download/OntixTutorialData.zip

After downloading:
- from the root of the repository, create the folders `data/raw` if not created yet
- move the donwloaded files there

#### Extra 2: Get correct path
We assume you are in the root of the package. The following code ensures that the correct paths are used.
[1] Tutorials/DeepDives/ConfigTutorial.ipynb


In [1]:
import os

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")


Changed to: /Users/maximilianjoas/development/autoencodix_package


In [3]:
import os
import autoencodix as acx
from autoencodix.configs.default_config import DataConfig, DataInfo, DataCase
from autoencodix.configs import OntixConfig

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
data_root = "data/raw"
rna_file = "combined_rnaseq_formatted.parquet"
meth_file = "combined_meth_formatted.parquet"
clin_file = "combined_clin_formatted.parquet"
ont_genelevel = "chromosome_ont_genelevel_ncbi.txt"
ont_hiddenlevel = "chromosome_ont_hiddenlevel.txt"
# ---------------------------------------------------------------------
# Define individual data modalities
# ---------------------------------------------------------------------
rna_info = DataInfo(
    file_path=os.path.join(data_root, rna_file),
    data_type="NUMERIC",
    filtering="VAR",
)

meth_info = DataInfo(
    file_path=os.path.join(data_root, meth_file),
    data_type="NUMERIC",
    filtering="VAR",
)

anno_info = DataInfo(
    file_path=os.path.join(data_root, clin_file),
    data_type="ANNOTATION",
)
# ---------------------------------------------------------------------
# Combine into DataConfig
# ---------------------------------------------------------------------
data_config = DataConfig(
    data_info={
        "RNA": rna_info,
        "METH": meth_info,
        "ANNO": anno_info,
    },
    annotation_columns=[
        # "CANCER_TYPE",
        "CANCER_TYPE_ACRONYM",
        # "TMB_NONSYNONYMOUS",
        # "AGE",
        # "OS_STATUS",
        # "GRADE",
        "SEX",
    ],
)

# ---------------------------------------------------------------------
# Define the full DefaultConfig (roughly equivalent to old cfg)
# ---------------------------------------------------------------------
ontix_config = OntixConfig(
    data_config=data_config,
    reproducible=True,
    global_seed=42,
    epochs=10,
    learning_rate=0.0005,
    batch_size=128,
    drop_p=0.3,
    k_filter=2000,
    latent_dim=6,
    device="cpu",
    reconstruction_loss="mse",
    default_vae_loss="kl",
    beta=0.001,
    save_memory=False,
    scaling="MINMAX",
    train_ratio=0.7,
    test_ratio=0.2,
    valid_ratio=0.1,
)

# ---------------------------------------------------------------------
# Now pass into your Ontix object
# ---------------------------------------------------------------------

ont_files = [ont_hiddenlevel, ont_genelevel]
ont_files = [os.path.join(data_root, f) for f in ont_files]
ontix = acx.Ontix(
    ontologies=ont_files,
    config=ontix_config,
)

In [4]:
result = ontix.run()

reading parquet: data/raw/combined_rnaseq_formatted.parquet
reading parquet: data/raw/combined_meth_formatted.parquet
reading parquet: data/raw/combined_clin_formatted.parquet
anno key: paired
Features in feature_order not found in all_feature_names: ['100133144', '10357', '10431', '155060', '390284', '57714', '645851', '653553', '729884', '246182', '119385', '653268', '728404', '200810', '138649', '441425', '728747', '729171', '23520', '303', '304', '305', '244', '375719', '441432', '503640', '641522', '432369', '92270', '6791', '85319', '606', '23629', '286076', '414235', '170393', '255352', '283422', '283416', '374467', '84837', '400223', '650662', '280655', '283651', '80035', '283687', '196968', '284185', '147429', '147525', '494514', '574036', '284573', '149469', '84791', '253868', '284836', '54094', '54067', '282566', '114041', '114043', '149992', '55267', '29798', '348738', '339942', '93556', '646450', '317648', '92070', '79614', '153571', '116349', '441108', '285679', '85411', 

##### Now we can call ontix.explain()
This will calculate gene-by-latent-dimension attribution scores. The explain step returns these as pd.DataFrame and also saves this DataFrame to `result.embedding_attributions`

In [5]:
latent_contributions = ontix.explain()

In [6]:
print(latent_contributions)

       latent_dimension_0  latent_dimension_1  latent_dimension_2  \
8755             0.000968            0.000409            0.002195   
6439             0.001164            0.001106            0.001054   
1277             0.000582            0.000512            0.000810   
3481             0.000087            0.000107            0.000175   
2335             0.000686            0.000498            0.001056   
...                   ...                 ...                 ...   
3241             0.001957            0.002347            0.002047   
2070             0.002633            0.003304            0.004940   
3199             0.002069            0.002361            0.002212   
56244            0.004850            0.004543            0.009528   
8354             0.002981            0.002822            0.003608   

       latent_dimension_3  latent_dimension_4  latent_dimension_5  
8755             0.001674            0.001555            0.000466  
6439             0.001835          

        
### 3) How to Customize the Explain Step 
TODO Vincent explain, besser
You have the follwong options:
```python
        method: Literal["DeepLiftShap", "IntegratedGradients"] = "DeepLiftShap",
        baseline_type: Literal["mean", "random_sample"] = "mean",
        baseline_group: str = "all",
        obs_col: Optional[str] = None,
        n_subset: int = 100,
        seed_int: int = 12,
        split: Literal["train", "test", "valid"] = "train",
        llm_explain: bool = False,
        llm_client: Literal["ollama", "mistral"] = "mistral",
        llm_model: str = "mistral-large-latest",
        top_n_genes: int = 40,
        prompt: str = PROMPT,
```

**More Details on the Options:**

```python
        """Runs the feature-importance explainer and returns gene-by-latent-dimension attribution scores.

        Args:
            method:  Specifies which attribution algorithm to use for explaining the model.
            baseline_type: Specifies whether the feature-importance algorithm should use the mean baseline or a random-sample baseline.
            baseline_group: Specifies whether the baseline is computed using all data (default) or which subset in obs_col to filter for.
            obs_col: If baseline_group is not 'all', this specifies the observation column used to filter the baseline data.
            n_subset: Specifies the number of cells to use when subsampling for the attribution computation.
            seed_int: Defines the random seed used for reproducible subsampling and attribution calculations.
            split: The split to use for feature importance calculation (train, valid, test), default is train.
            llm_explain: Whether to use LLM explainers for feature importance calculation.
            llm_client: The LLM client to use for feature importance calculation.
            llm_model: The LLM model to use for feature importance calculation.
            top_n_genes: How many top (contribution to embedding) genes to consider for LLM explanation.

        Returns:
            pd.DataFrame: The generated samples in the input space.
                A DataFrame of attribution scores with genes as the index and latent dimensions as columns.
                Each entry represents the contribution of a given gene to a specific latent dimension.
        """
        ```

### 4) How To Use LLM to Get Insights of Gene Contributions
We can send the gene attribution score directly to an LLM and get biological insights for each latent dimension, i.e. latent dim 1 is associated with cell cylce ..
The LLM will give you and explanation for each latent dim and will save this in an extra markdown file. Additionally, we get  parsable JSON that is stored in `result.embedding_explanations`.  

Before, we can obtain this, we need to set up our LLM provider (in the future this might be provided by use)
See below how to do  this:


#### 🧬 Gene Expression Explanation – README
You can get LLM explanations from our  `.explain` step by setting `llm_explain=True`. Therefore you need to setup LLM Clients

You can use either:

* **Mistral API** (cloud-based)
* **Ollama** (local models running on your machine)



#### Requirements

Depending on if you want to use Mistral or Ollama you need to have:
- a Mistral API key
- Ollam installed and at least one model served


#### Environment Setup

##### 1. Using Mistral API

Set an API key in your `.env` file in the root of the repository

```
MISTRAL_API_KEY=your_api_key_here
```

**Models**

You may use any model served by Mistral, for example:

* `mistral-small-latest`
* `mistral-medium-latest`
* `mistral-large-latest`

Make sure the `model_name` you pass to `.explain` matches an available Mistral model.


##### 2. Using Ollama

Ollama runs models **locally**.
You must first install Ollama and pull the model you want:

```bash
ollama pull <model-name>
```


Your `model_name` must match exactly the name of the model served by Ollama:

* `qwen2.5:0.5b`
* `deepseek-r1:8b`


##### 🧬 Using `.explain()` for Gene Expression Interpretation

The `.explain()` method can generate a short biological explanation and hypothesis about what is happening in disease vs healthy samples given a list of altered genes.

### Example


In [15]:

ontix.explain(llm_explain=True, llm_client="ollama", llm_model="qwen2.5:0.5b")

Saved explanations to: /Users/maximilianjoas/development/autoencodix_package/latent_explanations.md


,latent_dimension_0,latent_dimension_1,latent_dimension_2,latent_dimension_3,latent_dimension_4,latent_dimension_5
8755,0.000968,0.000409,0.002195,0.001674,0.001555,0.000466
6439,0.001164,0.001106,0.001054,0.001835,0.001380,0.001212
1277,0.000582,0.000512,0.000810,0.001507,0.000725,0.000543
3481,0.000087,0.000107,0.000175,0.000211,0.000197,0.000093
2335,0.000686,0.000498,0.001056,0.000816,0.001146,0.000446
...,...,...,...,...,...,...
3241,0.001957,0.002347,0.002047,0.002891,0.002869,0.004185
2070,0.002633,0.003304,0.004940,0.007683,0.002705,0.002184
3199,0.002069,0.002361,0.002212,0.003260,0.001887,0.002285
56244,0.004850,0.004543,0.009528,0.008237,0.009779,0.003268


**Parsable JSON Output**

In [17]:

result.embedding_explanations

{'latent_dimension_3': {'TLDR': 'The dominant biological themes represented by the provided gene set are likely to be related to signal transduction pathways, regulatory programs, and cellular states, as identified through the identification of top contributors using Captum. The hypothesized mechanisms include signal feedback loops, protein interaction networks, and dynamic signaling cascades.',
  'DETAILS': {'dominant_themes': 'signal transduction pathways, regulatory programs, cellular states',
   'hypotheses': ['signal feedback loops',
    'protein interaction networks',
    'dynamic signaling cascades']},
  'pathways_summary': 'The gene set identified as top contributors likely reflects a complex interplay of signal transduction pathways, regulatory network interactions, and dynamic signaling pathways involved in cellular functions such as growth, differentiation, response to stimuli, and regulation.'},
 'latent_dimension_2': {'TLDR': 'The latent dimension identified by Captum may 